# Temporal Sensor baseline stability

This notebook evaluates the sensor-only split-heart-rate GRU across ten training
seeds using the same fixed subject split and test-window eligibility as the
visual and fusion stability experiments.

The model receives motion/device statistics and heart rate, but no visual
features. Visual availability is used only to preserve the exact shared
evaluation cohort: sequences containing ten missing images are excluded, while
sequences containing zero to nine missing images remain eligible.


In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

from tqdm import tqdm
import json
import time
import os

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [38]:
df = pd.read_parquet("Data/dipser_dataset.parquet", engine="pyarrow")
df.head()

,group,time,time_sec,image_path,metadata,subject,experiment,self_labeling emotion,self_labeling attention,labeler_02 attention,labeler_04 attention,labeler_02 emotion,labeler_01 attention,labeler_03 emotion,labeler_03 attention,labeler_01 emotion,labeler_04 emotion,self_labeling emotionfilled,self_labeling attentionfilled,labeler_02 attentionfilled,labeler_04 attentionfilled,labeler_02 emotionfilled,labeler_01 attentionfilled,labeler_03 emotionfilled,labeler_03 attentionfilled,labeler_01 emotionfilled,labeler_04 emotionfilled,samsung_rotation_vector value0_mean,samsung_rotation_vector value0_std,samsung_rotation_vector value1_mean,samsung_rotation_vector value1_std,samsung_rotation_vector value2_mean,samsung_rotation_vector value2_std,samsung_rotation_vector value3_mean,samsung_rotation_vector value3_std,samsung_rotation_vector value4_mean,samsung_rotation_vector value4_std,lsm6dso_gyroscope value0_mean,lsm6dso_gyroscope value0_std,lsm6dso_gyroscope value1_mean,lsm6dso_gyroscope value1_std,lsm6dso_gyroscope value2_mean,lsm6dso_gyroscope value2_std,samsung_linear_acceleration_sensor value0_mean,samsung_linear_acceleration_sensor value0_std,samsung_linear_acceleration_sensor value1_mean,samsung_linear_acceleration_sensor value1_std,samsung_linear_acceleration_sensor value2_mean,samsung_linear_acceleration_sensor value2_std,opt3007_light value0_mean,opt3007_light value0_std,heart_rate,heart_rate_std,accel_magnitude_mean,accel_magnitude_std,gyro_magnitude_mean,gyro_magnitude_std,labeler_05 emotion,labeler_05 attention,labeler_05 emotionfilled,labeler_05 attentionfilled,invalid_reason,gender,age,race,subject_experiment_id,attention,subject_id,age_group
0,group01,2026-05-08 10:40:43.047895,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_047895.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_047895.json,subject_01,experiment01,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109524,0.000163,-0.659717,0.000588,0.632875,0.000495,0.390186,0.000218,246.0,0.0,-0.006022,0.006482,-0.003838,0.003665,-0.000864,0.006845,0.052984,0.034713,-0.132399,0.047077,0.017549,0.043508,63.0,0.0,78.0,NaN,0.155556,0.041540,0.011095,0.005498,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
1,group01,2026-05-08 10:40:43.195317,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_195317.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_195317.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109485,0.000109,-0.660015,0.000520,0.632620,0.000426,0.390106,0.000225,246.0,0.0,-0.004508,0.005235,-0.004068,0.003267,-0.001026,0.005944,0.053056,0.036979,-0.124020,0.045047,0.012282,0.043478,63.0,0.0,78.0,NaN,0.148634,0.038779,0.009572,0.004378,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
2,group01,2026-05-08 10:40:43.295405,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_295405.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_295405.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109478,0.000101,-0.660194,0.000514,0.632473,0.000406,0.390043,0.000252,246.0,0.0,-0.004581,0.005193,-0.003897,0.003153,-0.000525,0.005644,0.054420,0.036670,-0.129478,0.044805,0.015730,0.040968,63.0,0.0,78.0,NaN,0.153129,0.038995,0.009384,0.004079,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
3,group01,2026-05-08 10:40:43.395835,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_395835.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_395835.json,subject_01,experime

In [39]:
df.shape

(889685, 69)

In [40]:
SEED = 42
SEQUENCE_LENGTH = 10
MAX_MISSING_VISUAL_FRAMES = SEQUENCE_LENGTH -1  # remove a sequence only when all the images are unavailable

# Label-imbalance intervention toggles.
USE_WEIGHTED_LOSS = True
USE_WEIGHTED_SAMPLER = False
WEIGHT_ALPHA = 0.5       # 0=no weighting, 1=full inverse-frequency weighting
STRATIFY_COLUMN = 'age_group'
BATCH_SIZE = 128
NUM_WORKERS = 4

ATTENTION_BINS = [2, 2.5, 3, 3.5, 4.75]

In [41]:
df[['attention', 'image_path', 'age', 'gender']].isna().sum()

attention     0
image_path    0
age           0
gender        0
dtype: int64

In [42]:
sensor_cols = [
    col for col in df.columns
    if (
       # "sensor" in col.lower()
        "acceleration" in col.lower()
        or "gyro" in col.lower()
      #  or "rotation" in col.lower()
        or "heart" in col.lower()
      #  or "light" in col.lower()
        or "accel" in col.lower()
    )
    and "std" not in col.lower()
]

# Heart rate is physiologically different from motion/device sensors, so Fusion
# follows Temporal Sensor and gives it a separate projection stream.
hr_cols = ["heart_rate"] if "heart_rate" in sensor_cols else []
motion_cols = [col for col in sensor_cols if col not in hr_cols]
hr_indices = [sensor_cols.index(col) for col in hr_cols]
motion_indices = [sensor_cols.index(col) for col in motion_cols]

len(sensor_cols), len(motion_cols), len(hr_cols), sensor_cols[:5]

(9,
 8,
 1,
 ['lsm6dso_gyroscope value0_mean',
  'lsm6dso_gyroscope value1_mean',
  'lsm6dso_gyroscope value2_mean',
  'samsung_linear_acceleration_sensor value0_mean',
  'samsung_linear_acceleration_sensor value1_mean'])

In [43]:
# Treat physiologically impossible HR values as missing before missing flags and scaling.
# The smartwatch can emit 0, which should not be interpreted as a real heart rate.
df.loc[df["heart_rate"] < 30, "heart_rate"] = np.nan

df[sensor_cols] = df[sensor_cols].astype(np.float32)

In [44]:
# keep only 1 frame per second (the first)
df_sec = (df.sort_values(["subject_experiment_id", "time_sec"])
      .groupby(["subject_experiment_id", "time_sec"])
      .first()
      .reset_index())
df_sec.shape

(111754, 69)

In [45]:
reconstructed_sequences = []

for sequence_id, sequence_df in df_sec.groupby("subject_experiment_id"):
    sequence_df = sequence_df.sort_values("time_sec")

    complete_seconds = pd.DataFrame({
        "time_sec": np.arange(
            sequence_df["time_sec"].min(),
            sequence_df["time_sec"].max() + 1
        )
    })

    reconstructed = complete_seconds.merge(
        sequence_df,
        on="time_sec",
        how="left"
    )

    reconstructed["subject_experiment_id"] = sequence_id
    reconstructed_sequences.append(reconstructed)

temporal_frame_dataset = pd.concat(reconstructed_sequences, ignore_index=True)

sequence_metadata = (
    df[["subject_experiment_id", "subject_id", "gender", "age", "age_group"]]
    .drop_duplicates("subject_experiment_id"))

# Restore metadata for reconstructed missing seconds. Sensor/image/target values
# stay missing unless observed, so missingness flags remain meaningful.
temporal_frame_dataset = temporal_frame_dataset.drop(
    columns=["subject_id", "gender", "age", "age_group"],
    errors="ignore").merge(sequence_metadata, on="subject_experiment_id", how="left")

temporal_frame_dataset["visual_missing"] = temporal_frame_dataset["image_path"].isna().astype(np.float32)
temporal_frame_dataset["motion_missing"] = temporal_frame_dataset[motion_cols].isna().all(axis=1).astype(np.float32)
temporal_frame_dataset["hr_missing"] = temporal_frame_dataset[hr_cols].isna().all(axis=1).astype(np.float32) if hr_cols else 1.0
temporal_frame_dataset["sensor_missing"] = temporal_frame_dataset[sensor_cols].isna().all(axis=1).astype(np.float32)
temporal_frame_dataset["sensor_partial_nan"] = (
    temporal_frame_dataset[sensor_cols].isna().any(axis=1)
    & ~temporal_frame_dataset[sensor_cols].isna().all(axis=1)
).astype(np.float32)

temporal_frame_dataset[["subject_experiment_id", "time_sec", "visual_missing", "motion_missing", "hr_missing", "sensor_missing", "attention"]].head()

,subject_experiment_id,time_sec,visual_missing,motion_missing,hr_missing,sensor_missing,attention
0,group01_experiment01_subject_01,0,0.0,0.0,0.0,0.0,3.25
1,group01_experiment01_subject_01,1,0.0,0.0,0.0,0.0,3.00
2,group01_experiment01_subject_01,2,0.0,0.0,0.0,0.0,3.00
3,group01_experiment01_subject_01,3,0.0,0.0,0.0,0.0,3.25
4,group01_experiment01_subject_01,4,0.0,0.0,0.0,0.0,3.25


In [46]:
temporal_frame_dataset.shape

(121631, 74)

In [47]:
# Select one fixed age-stratified split using demographic metadata only.
# Model predictions, labels, and test performance are never used to choose it.
SPLIT_SEARCH_TRIALS = 1_000
MIN_MALE_VAL_SUBJECTS = 3
MIN_MALE_TEST_SUBJECTS = 3


def demographic_distance(split_df, full_df, column):
    categories = sorted(full_df[column].astype(str).unique())
    full_dist = full_df[column].astype(str).value_counts(normalize=True).reindex(categories, fill_value=0.0)
    split_dist = split_df[column].astype(str).value_counts(normalize=True).reindex(categories, fill_value=0.0)
    return float(np.abs(split_dist - full_dist).sum())


def find_constrained_age_stratified_split(subject_df, trials=SPLIT_SEARCH_TRIALS, seed=SEED):
    candidates = []
    all_age_groups = set(subject_df["age_group"].astype(str))

    for offset in range(trials):
        split_seed = seed + offset
        try:
            train_sub, temp_sub = train_test_split(
                subject_df,
                test_size=0.3,
                stratify=subject_df["age_group"],
                random_state=split_seed,
            )
            val_sub, test_sub = train_test_split(
                temp_sub,
                test_size=0.5,
                stratify=temp_sub["age_group"],
                random_state=split_seed,
            )
        except ValueError:
            continue

        val_males = int((val_sub["gender"].astype(str).str.lower() == "male").sum())
        test_males = int((test_sub["gender"].astype(str).str.lower() == "male").sum())
        val_has_all_ages = set(val_sub["age_group"].astype(str)) == all_age_groups
        test_has_all_ages = set(test_sub["age_group"].astype(str)) == all_age_groups

        representation_penalty = (
            max(0, MIN_MALE_VAL_SUBJECTS - val_males) * 100
            + max(0, MIN_MALE_TEST_SUBJECTS - test_males) * 100
            + (0 if val_has_all_ages else 100)
            + (0 if test_has_all_ages else 100)
        )
        balance_score = sum(
            demographic_distance(split, subject_df, column)
            for split in [train_sub, val_sub, test_sub]
            for column in ["age_group", "gender"]
        )
        candidates.append(
            (
                representation_penalty,
                balance_score,
                split_seed,
                train_sub.copy(),
                val_sub.copy(),
                test_sub.copy(),
            )
        )

    if not candidates:
        raise RuntimeError("Could not construct an age-stratified subject split.")

    best = min(candidates, key=lambda item: (item[0], item[1], item[2]))
    if best[0] > 0:
        print("WARNING: No split satisfied every requested representation constraint.")
    return best[3], best[4], best[5], best[2], best[0], best[1]


subject_df = (
    temporal_frame_dataset[["subject_id", "age_group", "gender"]]
    .dropna(subset=["subject_id", "age_group", "gender"])
    .drop_duplicates("subject_id")
    .copy()
)

train_sub, val_sub, test_sub, SPLIT_RANDOM_STATE, split_penalty, split_balance_score = (
    find_constrained_age_stratified_split(subject_df)
)

train_subjects = train_sub["subject_id"]
val_subjects = val_sub["subject_id"]
test_subjects = test_sub["subject_id"]

train_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(train_subjects)].copy()
val_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(val_subjects)].copy()
test_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(test_subjects)].copy()

# Train-only sensor normalization, identical to the original notebook.
sensor_means = train_frames.loc[train_frames["sensor_missing"] == 0, sensor_cols].mean()
sensor_stds = train_frames.loc[train_frames["sensor_missing"] == 0, sensor_cols].std()
sensor_stds = sensor_stds.replace(0, np.nan).fillna(1.0)
sensor_means = sensor_means.fillna(0.0)


def apply_sensor_scaling(frame_df):
    frame_df = frame_df.copy()
    scaled = ((frame_df[sensor_cols] - sensor_means) / sensor_stds).astype(np.float32)
    scaled = scaled.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    scaled.loc[frame_df["sensor_missing"] == 1, :] = 0.0
    frame_df.loc[:, sensor_cols] = scaled.to_numpy(dtype=np.float32)
    return frame_df


train_frames = apply_sensor_scaling(train_frames)
val_frames = apply_sensor_scaling(val_frames)
test_frames = apply_sensor_scaling(test_frames)

MOTION_INPUT_DIM = len(motion_cols) + 1
HR_INPUT_DIM = len(hr_cols) + 1
SENSOR_INPUT_DIM = len(sensor_cols) + 1


def split_demographic_table(split_df, split_name):
    rows = []
    for attribute in ["age_group", "gender"]:
        for group, count in split_df[attribute].astype(str).value_counts().sort_index().items():
            rows.append({
                "split": split_name,
                "attribute": attribute,
                "group": group,
                "subjects": int(count),
                "proportion": float(count / len(split_df)),
            })
    return pd.DataFrame(rows)


split_demographics = pd.concat(
    [
        split_demographic_table(train_sub, "train"),
        split_demographic_table(val_sub, "validation"),
        split_demographic_table(test_sub, "test"),
    ],
    ignore_index=True,
)

print(f"Selected demographic-only split random state: {SPLIT_RANDOM_STATE}")
print(f"Constraint penalty: {split_penalty}; demographic balance score: {split_balance_score:.4f}")
display(split_demographics)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_frames), len(val_frames), len(test_frames)],
    "subjects": [train_frames.subject_id.nunique(), val_frames.subject_id.nunique(), test_frames.subject_id.nunique()],
    "target_mean": [train_frames.attention.mean(), val_frames.attention.mean(), test_frames.attention.mean()],
    "visual_missing_rate": [train_frames.visual_missing.mean(), val_frames.visual_missing.mean(), test_frames.visual_missing.mean()],
    "motion_missing_rate": [train_frames.motion_missing.mean(), val_frames.motion_missing.mean(), test_frames.motion_missing.mean()],
    "hr_missing_rate": [train_frames.hr_missing.mean(), val_frames.hr_missing.mean(), test_frames.hr_missing.mean()],
    "sensor_missing_rate": [train_frames.sensor_missing.mean(), val_frames.sensor_missing.mean(), test_frames.sensor_missing.mean()],
})


Selected demographic-only split random state: 45
Constraint penalty: 0; demographic balance score: 0.3941


,split,attribute,group,subjects,proportion
0,train,age_group,"(13, 20]",13,0.333333
1,train,age_group,"(20, 22]",10,0.256410
2,train,age_group,"(22, 26]",10,0.256410
3,train,age_group,"(26, 44]",6,0.153846
4,train,gender,female,27,0.692308
5,train,gender,male,12,0.307692
6,validation,age_group,"(13, 20]",3,0.333333
7,validation,age_group,"(20, 22]",3,0.333333
8,validation,age_group,"(22, 26]",2,0.222222
9,validation,age_group,"(26, 44]",1,0.111111


,split,rows,subjects,target_mean,visual_missing_rate,motion_missing_rate,hr_missing_rate,sensor_missing_rate
0,train,84989,39,2.968285,0.086705,0.086705,0.095083,0.086705
1,val,19092,9,2.944667,0.070920,0.070920,0.074324,0.070920
2,test,17550,9,2.879391,0.065755,0.065755,0.114701,0.065755


In [48]:
def create_temporal_sequences(frame_df, sequence_length=SEQUENCE_LENGTH):
    sequences = []

    for sequence_id, sequence_df in frame_df.groupby("subject_experiment_id"):
        sequence_df = sequence_df.sort_values("time_sec").reset_index(drop=True)

        if len(sequence_df) < sequence_length:
            continue

        for i in range(sequence_length - 1, len(sequence_df)):
            target = sequence_df.iloc[i]["attention"]

            if pd.isna(target):
                continue

            history = sequence_df.iloc[i - sequence_length + 1:i + 1]
            visual_missing_flags = history["visual_missing"].astype(np.float32).values

            if visual_missing_flags.sum() > MAX_MISSING_VISUAL_FRAMES:
                continue

            motion_missing_flags = history["motion_missing"].astype(np.float32).values
            hr_missing_flags = history["hr_missing"].astype(np.float32).values
            sensor_missing_flags = history["sensor_missing"].astype(np.float32).values

            sequence_record = {
                "subject_experiment_id": sequence_id,
                "subject_id": sequence_df.iloc[i]["subject_id"],
                "gender": sequence_df.iloc[i]["gender"],
                "age": sequence_df.iloc[i]["age"],
                "age_group": sequence_df.iloc[i]["age_group"],
                "time_sec": int(sequence_df.iloc[i]["time_sec"]),
                "visual_missing_flags": visual_missing_flags.tolist(),
                "motion_missing_flags": motion_missing_flags.tolist(),
                "hr_missing_flags": hr_missing_flags.tolist(),
                "sensor_missing_flags": sensor_missing_flags.tolist(),
                "target": float(target),
            }

            # Store every scaled sensor as its own temporal column. Each value is
            # a length-SEQUENCE_LENGTH list, e.g. train_df["heart_rate"].iloc[0].
            for sensor_col in sensor_cols:
                sequence_record[sensor_col] = (
                    history[sensor_col]
                    .astype(np.float32)
                    .to_numpy(dtype=np.float32)
                    .tolist()
                )

            sequences.append(sequence_record)

    return pd.DataFrame(sequences)

In [49]:
train_df = create_temporal_sequences(train_frames)
val_df = create_temporal_sequences(val_frames)
test_df = create_temporal_sequences(test_frames)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
})

,split,sequences,subjects,target_mean
0,train,75189,39,2.971392
1,val,17208,9,2.948236
2,test,15894,9,2.881307


### Train Test Split

In [50]:
class TemporalSensorDataset(Dataset):

    def __init__(self, sequence_df):
        self.df = sequence_df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sensors = np.stack(
            [
                np.asarray(row[sensor_col], dtype=np.float32)
                for sensor_col in sensor_cols
            ],
            axis=1,
        )
        sensors = torch.tensor(sensors, dtype=torch.float32)
        sensors = torch.nan_to_num(sensors, nan=0.0, posinf=0.0, neginf=0.0)

        motion = sensors[:, motion_indices]
        motion_missing = torch.tensor(
            row["motion_missing_flags"], dtype=torch.float32
        ).unsqueeze(-1)
        motion = torch.cat([motion, motion_missing], dim=-1)

        if hr_indices:
            heart_rate = sensors[:, hr_indices]
        else:
            heart_rate = torch.zeros((sensors.shape[0], 0), dtype=torch.float32)
        hr_missing = torch.tensor(
            row["hr_missing_flags"], dtype=torch.float32
        ).unsqueeze(-1)
        heart_rate = torch.cat([heart_rate, hr_missing], dim=-1)

        target = torch.tensor(row["target"], dtype=torch.float32)
        sample_weight = torch.tensor(row["sample_weight"], dtype=torch.float32)
        return motion, heart_rate, target, sample_weight, idx


In [51]:
# Quantify target imbalance and create inverse-frequency train weights.
# Weights are learned from train only; val/test weights are diagnostic only.
def add_attention_bin_weights(alpha, train_df, val_df, test_df):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    train_bins = pd.cut(train_df["target"], bins=ATTENTION_BINS, include_lowest=True)
    bin_counts = train_bins.value_counts().sort_index()
    nonzero_counts = bin_counts[bin_counts > 0]

    bin_weights = (len(train_df) / (len(nonzero_counts) * nonzero_counts)) ** alpha

    for split_df in [train_df, val_df, test_df]:
        split_bins = pd.cut(split_df["target"], bins=ATTENTION_BINS, include_lowest=True)
        split_df["attention_bin"] = split_bins.astype(str)
        split_df["sample_weight"] = split_bins.map(bin_weights).astype(float).fillna(1.0)

    train_weight_mean = train_df["sample_weight"].mean()

    for split_df in [train_df, val_df, test_df]:
        split_df["sample_weight"] = split_df["sample_weight"] / train_weight_mean

    normalized_bin_weights = bin_weights / train_weight_mean

    imbalance_table = pd.DataFrame({
        "bin": bin_counts.index.astype(str),
        "train_count": bin_counts.values,
        "weight": [
            float(normalized_bin_weights.get(idx, np.nan))
            for idx in bin_counts.index
        ],
    })

    return train_df, val_df, test_df, imbalance_table

train_df, val_df, test_df, imbalance_table = add_attention_bin_weights(WEIGHT_ALPHA, train_df, val_df, test_df)
display(imbalance_table)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
    "mean_sample_weight": [train_df.sample_weight.mean(), val_df.sample_weight.mean(), test_df.sample_weight.mean()],
})

,bin,train_count,weight
0,"(1.999, 2.5]",15291,1.160744
1,"(2.5, 3.0]",35590,0.760835
2,"(3.0, 3.5]",18734,1.048671
3,"(3.5, 4.75]",5574,1.922521


,split,sequences,subjects,target_mean,mean_sample_weight
0,train,75189,39,2.971392,1.000000
1,val,17208,9,2.948236,0.981307
2,test,15894,9,2.881307,0.971843


In [52]:
train_df.shape, val_df.shape, test_df.shape

((75189, 22), (17208, 22), (15894, 22))

In [53]:
len(set(train_df['subject_experiment_id'])), len(set(val_df['subject_experiment_id'])) ,len(set(test_df['subject_experiment_id']))

(285, 64, 59)

In [54]:
train_dataset = TemporalSensorDataset(train_df)
val_dataset = TemporalSensorDataset(val_df)
test_dataset = TemporalSensorDataset(test_df)


def sensor_loader(dataset, shuffle=False, sampler=None):
    loader_kwargs = {
        "batch_size": BATCH_SIZE,
        "shuffle": shuffle if sampler is None else False,
        "sampler": sampler,
        "num_workers": NUM_WORKERS,
        "pin_memory": torch.cuda.is_available(),
        "persistent_workers": NUM_WORKERS > 0,
        "prefetch_factor": 4 if NUM_WORKERS > 0 else None,
    }
    loader_kwargs = {
        key: value for key, value in loader_kwargs.items() if value is not None
    }
    return DataLoader(dataset, **loader_kwargs)


if USE_WEIGHTED_SAMPLER:
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=torch.tensor(train_df["sample_weight"].values, dtype=torch.double),
        num_samples=len(train_df),
        replacement=True,
    )
    train_loader = sensor_loader(train_dataset, sampler=sampler)
else:
    train_loader = sensor_loader(train_dataset, shuffle=True)

val_loader = sensor_loader(val_dataset)
test_loader = sensor_loader(test_dataset)


### Training

In [55]:
class TemporalSplitHeartRateGRU(nn.Module):

    def __init__(
        self,
        motion_input_dim,
        hr_input_dim,
        motion_hidden_dim=64,
        hr_hidden_dim=16,
        dropout=0.3,
    ):
        super().__init__()
        self.motion_gru = nn.GRU(
            input_size=motion_input_dim,
            hidden_size=motion_hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.hr_gru = nn.GRU(
            input_size=hr_input_dim,
            hidden_size=hr_hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        fusion_dim = motion_hidden_dim + hr_hidden_dim
        self.regressor = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, motion, heart_rate):
        _, motion_hidden = self.motion_gru(motion)
        _, hr_hidden = self.hr_gru(heart_rate)
        features = torch.cat([motion_hidden[-1], hr_hidden[-1]], dim=1)
        return self.regressor(features).squeeze(1)


### Fixed-split multi-seed sensor baseline stability

The sensor-only split-heart-rate GRU is trained ten times on the exact same
fixed subject split. Overall regression performance, demographic worst-group
performance, subject-level stability, a diagnostic seed ensemble, and
subject-cluster bootstrap confidence intervals are reported.


In [31]:
import copy
import random
from importlib import reload
import src.evaluation as ev

ev = reload(ev)


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_model():
    return TemporalSplitHeartRateGRU(
        motion_input_dim=MOTION_INPUT_DIM,
        hr_input_dim=HR_INPUT_DIM,
        motion_hidden_dim=64,
        hr_hidden_dim=16,
        dropout=0.3,
    )


def make_train_loader():
    if USE_WEIGHTED_SAMPLER:
        sampler = torch.utils.data.WeightedRandomSampler(
            weights=torch.tensor(train_df["sample_weight"].values, dtype=torch.double),
            num_samples=len(train_df),
            replacement=True,
        )
        return sensor_loader(train_dataset, sampler=sampler)
    return sensor_loader(train_dataset, shuffle=True)


def weighted_task_loss(per_sample_loss, sample_weights):
    if USE_WEIGHTED_LOSS:
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)
    return per_sample_loss.mean()


def train_one_epoch_baseline(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    preds_all, labels_all = [], []

    for motion, heart_rate, labels, sample_weights, _idx in tqdm(loader, desc="Training", leave=False):
        motion = motion.to(device)
        heart_rate = heart_rate.to(device)
        labels = labels.to(device)
        sample_weights = sample_weights.to(device)

        preds = model(motion, heart_rate)
        loss = weighted_task_loss(criterion(preds, labels), sample_weights)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += float(loss.item())
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader),
        "mae": float(mean_absolute_error(labels_all, preds_all)),
        "rmse": float(np.sqrt(mean_squared_error(labels_all, preds_all))),
    }


def evaluate_loader(model, loader, sequence_df):
    model.eval()
    preds_all, labels_all, indices_all = [], [], []

    with torch.no_grad():
        for motion, heart_rate, labels, _sample_weights, idx in tqdm(loader, desc="Evaluating", leave=False):
            preds = model(motion.to(device), heart_rate.to(device))
            preds_all.extend(preds.detach().cpu().numpy())
            labels_all.extend(labels.detach().cpu().numpy())
            indices_all.extend(idx.detach().cpu().numpy())

    frame = sequence_df.iloc[np.asarray(indices_all, dtype=int)][
        ["subject_experiment_id", "subject_id", "time_sec", "attention_bin", "gender", "age", "age_group"]
    ].reset_index(drop=True)
    frame.insert(0, "true", np.asarray(labels_all, dtype=float))
    frame.insert(0, "pred", np.asarray(preds_all, dtype=float))

    overall = ev.compute_prediction_metrics(frame)
    age_mae, age_worst, age_gap = ev.compute_group_mae(frame, "age_group")
    gender_mae, gender_worst, gender_gap = ev.compute_group_mae(frame, "gender")
    return {
        "mae": overall["mae"],
        "rmse": overall["rmse"],
        "r2": overall["r2"],
        "true_mean": overall["true_mean"],
        "pred_mean": overall["pred_mean"],
        "age_worst_group_mae": age_worst,
        "age_gap": age_gap,
        "gender_worst_group_mae": gender_worst,
        "gender_gap": gender_gap,
        "age_mae_per_group": age_mae.to_dict(),
        "gender_mae_per_group": gender_mae.to_dict(),
    }, frame


In [32]:
class EarlyStopping:
    def __init__(self, patience, model_path):
        self.patience = patience
        self.model_path = model_path
        self.best_score = float("inf")
        self.best_epoch = None
        self.counter = 0

    def step(self, val_mae, model, epoch):
        if val_mae < self.best_score:
            self.best_score = float(val_mae)
            self.best_epoch = epoch
            self.counter = 0
            torch.save(copy.deepcopy(model.state_dict()), self.model_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


LOSS_TYPE = "mse"
criterion = nn.MSELoss(reduction="none")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 50
PATIENCE = 7
TRAIN_SHUFFLE = True

# Ten independent initialization/training seeds on the exact same subject split.
RUN_SEEDS = [42, 100, 2000, 2025, 2026, 2027, 2048, 4096, 7000, 8192]

RESULTS_DIR = "results/Sensor Baseline"
MODEL_DIR = "models/Sensor Baseline"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Fixed split random state: {SPLIT_RANDOM_STATE}")
print(f"Training shuffle: {TRAIN_SHUFFLE}")
print(f"Training seeds: {RUN_SEEDS}")

Fixed split random state: 47
Training shuffle: True
Training seeds: [42, 100, 2000, 2025, 2026, 2027, 2048, 4096, 7000, 8192]


In [33]:
run_records = []
test_predictions = {}

for run_seed in RUN_SEEDS:
    set_global_seed(run_seed)
    run_name = f"sensor_baseline_fixed_age_split_seed{run_seed}"
    model_path = os.path.join(MODEL_DIR, f"{run_name}.pt")

    model = make_model().to(device)
    optimizer = torch.optim.Adam(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.3, patience=3
    )
    early_stopping = EarlyStopping(PATIENCE, model_path)
    run_train_loader = make_train_loader()
    history = []

    print(f"\n=== {run_name} ===")
    for epoch in range(NUM_EPOCHS):
        train_metrics = train_one_epoch_baseline(model, run_train_loader, optimizer, criterion)
        val_metrics, _ = evaluate_loader(model, val_loader, val_df)
        scheduler.step(val_metrics["mae"])

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_mae": train_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "val_rmse": val_metrics["rmse"],
            "val_r2": val_metrics["r2"],
        })
        print(
            f"Epoch {epoch + 1:02d} | train MAE {train_metrics['mae']:.4f} | "
            f"val MAE {val_metrics['mae']:.4f} | val R2 {val_metrics['r2']:.4f}"
        )
        if early_stopping.step(val_metrics["mae"], model, epoch):
            print("Early stopping triggered")
            break

    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    val_metrics, val_predictions = evaluate_loader(model, val_loader, val_df)
    test_metrics, test_frame = evaluate_loader(model, test_loader, test_df)
    test_predictions[run_name] = test_frame

    val_prediction_path = os.path.join(RESULTS_DIR, f"{run_name}_val_predictions.csv")
    test_prediction_path = os.path.join(RESULTS_DIR, f"{run_name}_test_predictions.csv")
    ev.save_prediction_frame(val_predictions, val_prediction_path)
    ev.save_prediction_frame(test_frame, test_prediction_path)

    run_records.append({
        "run_name": run_name,
        "run_seed": run_seed,
        "best_epoch": early_stopping.best_epoch + 1,
        "num_epochs_run": len(history),
        "model_path": model_path,
        "val_prediction_path": val_prediction_path,
        "test_prediction_path": test_prediction_path,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "history": history,
    })

print("Finished all sensor baseline runs.")



=== sensor_baseline_fixed_age_split_seed42 ===


Epoch 01 | train MAE 1.1850 | val MAE 0.3721 | val R2 -0.2127


Epoch 02 | train MAE 0.5123 | val MAE 0.3621 | val R2 -0.1023


Epoch 03 | train MAE 0.4689 | val MAE 0.3526 | val R2 -0.0379


Epoch 04 | train MAE 0.4433 | val MAE 0.3511 | val R2 -0.0268


...

Epoch 16 | train MAE 0.3083 | val MAE 0.3611 | val R2 -0.0392


Epoch 17 | train MAE 0.3043 | val MAE 0.3600 | val R2 -0.0448


Epoch 18 | train MAE 0.3033 | val MAE 0.3563 | val R2 -0.0306


Epoch 19 | train MAE 0.3012 | val MAE 0.3595 | val R2 -0.0388
Early stopping triggered


Finished all sensor baseline runs.


In [34]:
metric_rows = []
for run in run_records:
    row = {
        "run_name": run["run_name"],
        "run_seed": run["run_seed"],
        "best_epoch": run["best_epoch"],
        "num_epochs_run": run["num_epochs_run"],
    }
    row.update({f"val_{key}": value for key, value in run["val_metrics"].items() if not isinstance(value, dict)})
    row.update({f"test_{key}": value for key, value in run["test_metrics"].items() if not isinstance(value, dict)})
    metric_rows.append(row)

seed_results = pd.DataFrame(metric_rows)

summary_metrics = [
    "test_mae",
    "test_rmse",
    "test_r2",
    "test_age_worst_group_mae",
    "test_age_gap",
    "test_gender_worst_group_mae",
    "test_gender_gap",
]
seed_summary = (
    seed_results[summary_metrics]
    .agg(["mean", "std", "min", "median", "max"])
    .T
    .reset_index(names="metric")
)

display(seed_results.round(4))
display(seed_summary.round(4))

,run_name,run_seed,best_epoch,num_epochs_run,val_mae,val_rmse,val_r2,val_true_mean,val_pred_mean,val_age_worst_group_mae,val_age_gap,val_gender_worst_group_mae,val_gender_gap,test_mae,test_rmse,test_r2,test_true_mean,test_pred_mean,test_age_worst_group_mae,test_age_gap,test_gender_worst_group_mae,test_gender_gap
0,sensor_baseline_fixed_age_split_seed42,42,13,20,0.3419,0.4282,0.0428,2.9653,2.9657,0.3771,0.0612,0.3513,0.0156,0.3368,0.4322,-0.0506,3.0175,2.9535,0.3782,0.1049,0.3550,0.0256
1,sensor_baseline_fixed_age_split_seed100,100,6,13,0.3392,0.4247,0.0584,2.9653,2.9368,0.3856,0.0696,0.3490,0.0162,0.3295,0.4220,-0.0017,3.0175,2.9492,0.3884,0.1365,0.3434,0.0195
2,sensor_baseline_fixed_age_split_seed2000,2000,10,17,0.3281,0.4112,0.1172,2.9653,2.9554,0.3641,0.0675,0.3332,0.0084,0.3353,0.4310,-0.0450,3.0175,2.9442,0.3918,0.1168,0.3366,0.0045
3,sensor_baseline_fixed_age_split_seed2025,2025,7,14,0.3464,0.4391,-0.0070,2.9653,2.9557,0.3969,0.0766,0.3541,0.0127,0.3464,0.4427,-0.1022,3.0175,2.9299,0.4084,0.1177,0.3732,0.0379
4,sensor_baseline_fixed_age_split_seed2026,2026,7,14,0.3389,0.4303,0.0333,2.9653,2.9316,0.3898,0.0727,0.3443,0.0088,0.3414,0.4381,-0.0794,3.0175,2.9409,0.3831,0.1036,0.3701,0.0403
5,sensor_baseline_fixed_age_split_seed2027,2027,6,13,0.3433,0.4336,0.0182,2.9653,2.9248,0.3801,0.0651,0.3497,0.0105,0.3364,0.4310,-0.0449,3.0175,2.9355,0.3850,0.1050,0.3476,0.0157
6,sensor_baseline_fixed_age_split_seed2048,2048,8,15,0.3433,0.4345,0.0141,2.9653,2.9456,0.3849,0.0793,0.3504,0.0117,0.3397,0.4360,-0.0694,3.0175,2.9272,0.3846,0.1021,0.3590,0.0272
7,sensor_baseline_fixed_age_split_seed4096,4096,2,9,0.3454,0.4421,-0.0204,2.9653,2.8658,0.3912,0.0679,0.3473,0.0048,0.3496,0.4509,-0.1436,3.0175,2.8817,0.4065,0.1306,0.3714,0.0306
8,sensor_baseline_fixed_age_split_seed7000,7000,7,14,0.3384,0.4289,0.0397,2.9653,2.9426,0.3828,0.0687,0.3488,0.0173,0.3360,0.4295,-0.0377,3.0175,2.9279,0.3826,0.1044,0.3596,0.0332
9,sensor_baseline_fixed_age_split_seed8192,8192,12,19,0.3420,0.4321,0.0253,2.9653,2.9275,0.3898,0.0668,0.3492,0.0120,0.3366,0.4341,-0.0601,3.0175,2.9331,0.3907,0.1110,0.3384,0.0060


,metric,mean,std,min,median,max
0,test_mae,0.3388,0.0058,0.3295,0.3367,0.3496
1,test_rmse,0.4347,0.0079,0.4220,0.4331,0.4509
2,test_r2,-0.0634,0.0388,-0.1436,-0.0553,-0.0017
3,test_age_worst_group_mae,0.3900,0.0101,0.3782,0.3867,0.4084
4,test_age_gap,0.1133,0.0121,0.1021,0.1080,0.1365
5,test_gender_worst_group_mae,0.3554,0.0136,0.3366,0.3570,0.3732
6,test_gender_gap,0.0241,0.0124,0.0045,0.0264,0.0403


In [35]:
mean_baseline_predictions = ev.make_mean_baseline_predictions(train_df, test_df)
mean_baseline_metrics = ev.compute_prediction_metrics(mean_baseline_predictions)

# Diagnostic seed ensemble: average predictions from all independently trained
# models. This is not treated as a single-model baseline.
ensemble_frame = next(iter(test_predictions.values())).copy()
ensemble_frame["pred"] = np.mean(
    [frame["pred"].to_numpy(dtype=float) for frame in test_predictions.values()],
    axis=0,
)
ensemble_metrics = ev.compute_prediction_metrics(ensemble_frame)
_, ensemble_age_worst, ensemble_age_gap = ev.compute_group_mae(ensemble_frame, "age_group")
_, ensemble_gender_worst, ensemble_gender_gap = ev.compute_group_mae(ensemble_frame, "gender")
ensemble_metrics.update({
    "age_worst_group_mae": ensemble_age_worst,
    "age_gap": ensemble_age_gap,
    "gender_worst_group_mae": ensemble_gender_worst,
    "gender_gap": ensemble_gender_gap,
})

reference_table = pd.DataFrame([
    {"model": "train-mean constant baseline", **mean_baseline_metrics},
    {"model": "ten-seed prediction ensemble (diagnostic)", **ensemble_metrics},
])

display(reference_table.round(4))


,model,n_samples,mae,rmse,r2,tolerant_accuracy,tolerance,one_off_accuracy,binary_threshold,binary_accuracy,binary_f1,binary_roc_auc,binary_confusion_matrix,true_mean,pred_mean,age_worst_group_mae,age_gap,gender_worst_group_mae,gender_gap
0,train-mean constant baseline,17673,0.3319,0.4287,-0.0338,0.2519,0.15,0.9999,3.0,0.3926,0.0000,0.5000,"[[6938, 0], [10735, 0]]",3.0175,2.9399,NaN,NaN,NaN,NaN
1,ten-seed prediction ensemble (diagnostic),17673,0.3309,0.4248,-0.0153,0.2984,0.15,0.9995,3.0,0.5109,0.4629,0.5798,"[[5306, 1632], [7011, 3724]]",3.0175,2.9323,0.3799,0.1127,0.3467,0.0224


In [36]:
BOOTSTRAP_RUNS = 1000
bootstrap_summary, _bootstrap_samples = ev.bootstrap_table(
    test_predictions,
    cluster_col="subject_id",
    n_boot=BOOTSTRAP_RUNS,
    seed=SEED,
)

# Quantify how much each test subject's MAE    
subject_rows = []
for run_name, frame in test_predictions.items():
    per_subject = (
        frame.assign(abs_error=np.abs(frame["true"].astype(float) - frame["pred"].astype(float)))
        .groupby("subject_id", observed=True)
        .agg(
            samples=("abs_error", "size"),
            mae=("abs_error", "mean"),
            true_mean=("true", "mean"),
            pred_mean=("pred", "mean"),
        )
        .reset_index()
    )
    per_subject.insert(0, "run_name", run_name)
    subject_rows.append(per_subject)

subject_seed_metrics = pd.concat(subject_rows, ignore_index=True)
subject_stability = (
    subject_seed_metrics.groupby("subject_id", observed=True)
    .agg(
        samples=("samples", "first"),
        true_mean=("true_mean", "first"),
        mean_mae=("mae", "mean"),
        sd_mae=("mae", "std"),
        min_mae=("mae", "min"),
        max_mae=("mae", "max"),
    )
    .reset_index()
    .sort_values("mean_mae", ascending=False)
)

display(bootstrap_summary.round(4))
display(subject_stability.round(4))


,model,metric,mean,ci_low,ci_high
0,sensor_baseline_fixed_age_split_seed42,mae,0.3362,0.2845,0.3986
1,sensor_baseline_fixed_age_split_seed42,rmse,0.4302,0.3760,0.4878
2,sensor_baseline_fixed_age_split_seed42,r2,-0.0824,-0.2354,0.0149
3,sensor_baseline_fixed_age_split_seed42,gender_gap,0.0506,0.0000,0.1261
4,sensor_baseline_fixed_age_split_seed42,gender_worst_group_mae,0.3617,0.2958,0.4188
...,...,...,...,...,...
65,sensor_baseline_fixed_age_split_seed8192,r2,-0.0898,-0.2356,0.0133
66,sensor_baseline_fixed_age_split_seed8192,gender_gap,0.0451,0.0007,0.1303
67,sensor_baseline_fixed_age_split_seed8192,gender_worst_group_mae,0.3537,0.2961,0.4308
68,sensor_baseline_fixed_age_split_seed8192,age_gap,0.1307,0.0426,0.2437


,subject_id,samples,true_mean,mean_mae,sd_mae,min_mae,max_mae
8,group03_subject_16,2385,3.2783,0.5102,0.0157,0.4853,0.5372
6,group03_subject_12,2488,3.2776,0.4150,0.0277,0.3747,0.4557
4,group02_subject_12,1967,2.6833,0.3756,0.0125,0.3588,0.3949
0,group01_subject_09,1813,2.7774,0.3230,0.0169,0.3069,0.3612
3,group02_subject_04,2178,3.1129,0.2820,0.0125,0.2553,0.3029
2,group02_subject_03,1351,2.9180,0.2743,0.0129,0.2463,0.2900
5,group03_subject_05,2419,2.9423,0.2666,0.0109,0.2527,0.2813
7,group03_subject_13,2239,3.0126,0.2604,0.0112,0.2436,0.2790
1,group02_subject_01,833,2.9481,0.2416,0.0155,0.2200,0.2727


In [37]:
split_demographics.to_csv(os.path.join(RESULTS_DIR, "split_demographics.csv"), index=False)
seed_results.to_csv(os.path.join(RESULTS_DIR, "seed_results.csv"), index=False)
seed_summary.to_csv(os.path.join(RESULTS_DIR, "seed_summary.csv"), index=False)
reference_table.to_csv(os.path.join(RESULTS_DIR, "reference_baselines.csv"), index=False)
bootstrap_summary.to_csv(os.path.join(RESULTS_DIR, "subject_bootstrap_summary.csv"), index=False)
subject_seed_metrics.to_csv(os.path.join(RESULTS_DIR, "subject_seed_metrics.csv"), index=False)
subject_stability.to_csv(os.path.join(RESULTS_DIR, "subject_stability.csv"), index=False)
ev.save_prediction_frame(ensemble_frame, os.path.join(RESULTS_DIR, "seed_ensemble_test_predictions.csv"))


def json_safe(value):
    """Recursively convert pandas/NumPy objects and non-JSON dictionary keys."""
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value
    
with open(os.path.join(RESULTS_DIR, "run_manifest.json"), "w") as file:
    json.dump(
        json_safe({
            "experiment": "Temporal Sensor fixed constrained split baseline stability",
            "split_stratify": STRATIFY_COLUMN,
            "split_random_state": SPLIT_RANDOM_STATE,
            "split_constraint_penalty": split_penalty,
            "split_balance_score": split_balance_score,
            "run_seeds": RUN_SEEDS,
            "loss_type": LOSS_TYPE,
            "early_stopping_metric": "mae",
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "sensor_cols": sensor_cols,
            "motion_cols": motion_cols,
            "hr_cols": hr_cols,
            "mean_baseline_metrics": mean_baseline_metrics,
            "ensemble_metrics": ensemble_metrics,
            "runs": run_records,
        }),
        file,
        indent=2,
        default=str,
    )
print(f"Saved sensor baseline stability results to: {RESULTS_DIR}")

Saved sensor baseline stability results to: results/Sensor Baseline


### Reading the results

Use `seed_summary.csv` to judge whether the fixed split produces consistently
acceptable predictive performance. The mean, standard deviation, and worst seed
matter more than the single best seed.

The fixed split should only be used for the later fairness experiment if its
baseline consistently beats the train-mean constant predictor and its MAE/R2
remain acceptable across seeds.
